Load Required Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import sqlalchemy
import tensorflow as tf
import psycopg


from sqlalchemy import create_engine, text
from getpass import getpass

from pathlib import Path


In [14]:
%pip install pandas numpy matplotlib seaborn scikit-learn sqlalchemy "psycopg[binary]" jupyter

   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ------------------------- -------------- 2.4/3.7 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 10.4 MB/s  0:00:00

   ---------------------------------------- 0/4 [psycopg-binary]
   ---------- ----------------------------- 1/4 [psycopg]
   ---------- ----------------------------- 1/4 [psycopg]
   ---------- ----------------------------- 1/4 [psycopg]
   ---------- ----------------------------- 1/4 [psycopg]
   ---------- ----------------------------- 1/4 [psycopg]
   -------------------- ------------------- 2/4 [jupyter-console]
   ---------------------------------------- 4/4 [jupyter]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
%pip install sqlalchemy

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------------------- -------------- 1.6/2.4 MB 9.9 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 8.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("TensorFlow:", tf.__version__)

Pandas: 3.0.3
NumPy: 2.4.6
Scikit-learn: 1.9.0
TensorFlow: 2.21.0


In [5]:
DATA_DIR = Path("../data/raw")
csv_path = DATA_DIR / "diabetic_data.csv"

In [6]:
df = pd.read_csv(
    csv_path,
    dtype=str,
    keep_default_na=False
)

In [7]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Dataset shape: (101766, 50)

Column names:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [8]:
print("\nFirst five rows:")
display(df.head())


First five rows:


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [9]:
print("\nData types:")
print(df.dtypes)


Data types:
encounter_id                str
patient_nbr                 str
race                        str
gender                      str
age                         str
weight                      str
admission_type_id           str
discharge_disposition_id    str
admission_source_id         str
time_in_hospital            str
payer_code                  str
medical_specialty           str
num_lab_procedures          str
num_procedures              str
num_medications             str
number_outpatient           str
number_emergency            str
number_inpatient            str
diag_1                      str
diag_2                      str
diag_3                      str
number_diagnoses            str
max_glu_serum               str
A1Cresult                   str
metformin                   str
repaglinide                 str
nateglinide                 str
chlorpropamide              str
glimepiride                 str
acetohexamide               str
glipizide                  

In [10]:
print("\nReadmission labels:")
print(df["readmitted"].value_counts(dropna=False))


Readmission labels:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


Load the cohort in Python and verify it

In [11]:

cohort_path = Path(
    "../data/processed/readmission_cohort.csv"
)

cohort = pd.read_csv(cohort_path)

In [12]:
print("Cohort shape:", cohort.shape)
print("\nColumns:")
print(cohort.columns.tolist())

print("\nTarget distribution:")
print(
    cohort["readmitted_within_30_days"]
    .value_counts(dropna=False)
)

print("\nTarget proportions:")
print(
    cohort["readmitted_within_30_days"]
    .value_counts(normalize=True, dropna=False)
)

print("\nDuplicate encounter IDs:",
      cohort["encounter_id"].duplicated().sum())

display(cohort.head())

Cohort shape: (101766, 23)

Columns:
['encounter_id', 'patient_nbr', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'max_glu_serum', 'a1cresult', 'diag_1', 'diag_2', 'diag_3', 'readmitted', 'race', 'gender', 'age', 'readmitted_within_30_days']

Target distribution:
readmitted_within_30_days
0    90409
1    11357
Name: count, dtype: int64

Target proportions:
readmitted_within_30_days
0    0.888401
1    0.111599
Name: proportion, dtype: float64

Duplicate encounter IDs: 0


,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,max_glu_serum,a1cresult,diag_1,diag_2,diag_3,readmitted,race,gender,age,readmitted_within_30_days
0,2278392,8222157,6,25,1,1,41,0,1,0,...,NaN,NaN,250.83,NaN,NaN,NO,Caucasian,Female,[0-10),0
1,149190,55629189,1,1,7,3,59,0,18,0,...,NaN,NaN,276,250.01,255,>30,Caucasian,Female,[10-20),0
2,64410,86047875,1,1,7,2,11,5,13,2,...,NaN,NaN,648,250,V27,NO,AfricanAmerican,Female,[20-30),0
3,500364,82442376,1,1,7,2,44,1,16,0,...,NaN,NaN,8,250.43,403,NO,Caucasian,Male,[30-40),0
4,16680,42519267,1,1,7,1,51,0,8,0,...,NaN,NaN,197,157,250,NO,Caucasian,Male,[40-50),0


In [16]:
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("PostgreSQL driver: installed")

Pandas: 3.0.3
Scikit-learn: 1.9.0
SQLAlchemy: 2.1.4
PostgreSQL driver: installed


In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL  # <-- Import URL helper
from getpass import getpass

db_password = getpass("PostgreSQL password: ")

# Safely builds the URI and automatically escapes special characters
connection_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password=db_password,
    host="localhost",
    port=5432,
    database="healthcare_readmission"
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())
